# 02 · Layer 1：Tools — 讓 Agent 真的能做事

LLM 本質上只會做一件事：**猜下一個字**。它不會查資料庫、不會呼叫 API、
不知道現在幾點。工具（Tool）就是補上這一塊的東西。

ADK 的設計很直接：**一個 Python 函式就是一個工具**。你不用寫 JSON Schema，
ADK 會從你的型別註記和 docstring 自動生成。

## 0. 環境

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

## 1. 沒有工具的時候

先看看問題長什麼樣子。問一個沒有工具的 agent「現在幾點」：

In [2]:
import datetime

from google.adk.agents import LlmAgent

blind = LlmAgent(
    name="blind_bot",
    model=get_model(),
    instruction="你是助理，用繁體中文簡短回答。",
)

print("實際時間 :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))
print("模型說的 :", await run_once(blind, "現在的日期和時間是幾點？"))

實際時間 : 2026-09-04 01:41


模型說的 : 現在的時間是 2024年5月17日 下午 4:30（UTC）。


模型要嘛老實說不知道、要嘛編一個給你。它沒有時鐘。

## 2. 一個函式 = 一個工具

把函式丟進 `tools=[]` 就成了。ADK 會讀三樣東西：

| 來源 | 變成 schema 的什麼 |
|---|---|
| 函式名稱 | 工具名稱 |
| 型別註記 `city: str` | 參數型別 |
| docstring | 工具說明 + 各參數說明 |

**模型只看得到這三樣東西**。你的函式內容它一無所知。

In [3]:
def get_current_time(timezone: str = "Asia/Taipei") -> dict:
    """取得指定時區的目前日期與時間。

    Args:
        timezone: IANA 時區名稱，例如 'Asia/Taipei'、'UTC'。
    """
    from zoneinfo import ZoneInfo

    now = datetime.datetime.now(ZoneInfo(timezone))
    return {"timezone": timezone, "datetime": now.strftime("%Y-%m-%d %H:%M:%S")}


clock_bot = LlmAgent(
    name="clock_bot",
    model=get_model(),
    instruction="你是助理。被問到時間時一定要呼叫 get_current_time，不要自己猜。",
    tools=[get_current_time],
)

print(await run_once(clock_bot, "現在台北時間幾點？", trace=True))

  🔧 [clock_bot] 呼叫 get_current_time({'timezone': 'Asia/Taipei'})
  ↩️  [clock_bot] get_current_time 回傳 {'timezone': 'Asia/Taipei', 'datetime': '2026-09-04 09:42:01'}


  💬 [clock_bot] 現在台北時間是 2026年9月4日 早上 9點42分。
現在台北時間是 2026年9月4日 早上 9點42分。


## 3. 看看 ADK 到底送了什麼給模型

這一步很值得做一次。你會發現 **docstring 是真的會被送出去的 API 文件**，
不是給人看的註解。

In [4]:
from google.adk.tools import FunctionTool

declaration = FunctionTool(func=get_current_time)._get_declaration()

print("工具名稱 :", declaration.name)
print("\n工具說明 :")
print(declaration.description)
print("\n參數 schema :")
import json

print(json.dumps(declaration.parameters_json_schema, indent=2, ensure_ascii=False))

工具名稱 : get_current_time

工具說明 :
取得指定時區的目前日期與時間。

Args:
    timezone: IANA 時區名稱，例如 'Asia/Taipei'、'UTC'。

參數 schema :
{
  "properties": {
    "timezone": {
      "default": "Asia/Taipei",
      "title": "Timezone",
      "type": "string"
    }
  },
  "title": "get_current_timeParams",
  "type": "object"
}


對照一下：`Args:` 區塊裡的說明會進到參數描述，型別註記變成 `type`，
有預設值的參數不會出現在 `required` 裡。

所以「工具沒被呼叫」或「參數傳錯」的第一個檢查點永遠是：**docstring 寫清楚了嗎？**

### 反例：沒有 docstring 的工具

In [5]:
def calc(a, b, op):  # 沒有型別註記、沒有 docstring
    if op == "add":
        return {"r": a + b}
    return {"r": a - b}


bad_declaration = FunctionTool(func=calc)._get_declaration()
print("說明 :", repr(bad_declaration.description))
print("參數 :", json.dumps(bad_declaration.parameters_json_schema, ensure_ascii=False))

說明 : None
參數 : {"properties": {"a": {"title": "A"}, "b": {"title": "B"}, "op": {"title": "Op"}}, "required": ["a", "b", "op"], "title": "calcParams", "type": "object"}


模型看到的是一個叫 `calc`、沒有任何說明、參數型別不明的東西。
它要嘛不敢用，要嘛亂傳參數。

## 4. 多個工具：模型自己選

給它三個工具，看它怎麼挑。注意 `trace=True` 印出來的呼叫順序。

In [6]:
_STOCK = {"2330": ("台積電", 1085), "2317": ("鴻海", 203), "AAPL": ("Apple", 268)}
_RATE = {"USD": 31.2, "JPY": 0.21, "EUR": 34.0}


def get_stock_price(symbol: str) -> dict:
    """查詢股票的目前價格。

    Args:
        symbol: 股票代號，例如 '2330'、'AAPL'。
    """
    if symbol not in _STOCK:
        return {"error": f"查無代號 {symbol}"}
    name, price = _STOCK[symbol]
    return {"symbol": symbol, "name": name, "price": price}


def convert_currency(amount: float, from_currency: str) -> dict:
    """把外幣金額換算成新台幣。

    Args:
        amount: 金額。
        from_currency: 原幣別代碼，例如 'USD'、'JPY'。
    """
    rate = _RATE.get(from_currency.upper())
    if rate is None:
        return {"error": f"不支援的幣別 {from_currency}"}
    return {"twd": round(amount * rate, 2), "rate": rate}


def get_current_weather(city: str) -> dict:
    """查詢城市目前天氣。

    Args:
        city: 城市名稱。
    """
    return {"city": city, "temp_c": 26, "condition": "多雲"}


analyst = LlmAgent(
    name="analyst",
    model=get_model(),
    instruction=(
        "你是金融助理。需要資料時一律呼叫工具取得，絕對不要自己編數字。"
        "用繁體中文回答。"
    ),
    tools=[get_stock_price, convert_currency, get_current_weather],
)

print(await run_once(analyst, "台積電現在多少錢？", trace=True))

  🔧 [analyst] 呼叫 get_stock_price({'symbol': '2330'})
  ↩️  [analyst] get_stock_price 回傳 {'symbol': '2330', 'name': '台積電', 'price': 1085}


  💬 [analyst] 台積電（2330）目前的價格是 **1085** 元。
台積電（2330）目前的價格是 **1085** 元。


### 一個問題觸發多個工具

「我有 500 美元，能買幾張 Apple？」需要 **先查匯率、再查股價**，
而且第二次呼叫要用到第一次的結果。模型會自己排順序。

In [7]:
print(await run_once(analyst, "我有 500 美元，換成台幣是多少？夠買幾股 Apple？", trace=True))

  🔧 [analyst] 呼叫 convert_currency({'from_currency': 'USD', 'amount': 500})
  🔧 [analyst] 呼叫 get_stock_price({'symbol': 'AAPL'})
  ↩️  [analyst] convert_currency 回傳 {'twd': 15600.0, 'rate': 31.2}
  ↩️  [analyst] get_stock_price 回傳 {'symbol': 'AAPL', 'name': 'Apple', 'price': 268}


  💬 [analyst] 500 美元換算成台幣是 **15,600** 元（匯率：1 USD = 31.2 TWD）。

Apple（AAPL）目前的股價是 **268** 美元。
用 500 美元可以買下 **1** 股（花費 268 美元），剩餘 232 美元。
500 美元換算成台幣是 **15,600** 元（匯率：1 USD = 31.2 TWD）。

Apple（AAPL）目前的股價是 **268** 美元。
用 500 美元可以買下 **1** 股（花費 268 美元），剩餘 232 美元。


## 5. `ToolContext`：讓工具讀寫 session state

到目前為止工具都是「純函式」——同樣輸入永遠同樣輸出。但實務上工具常常需要
知道「這個使用者是誰」、或是把結果**留給後面的 agent 用**。

只要在函式簽章加上 `tool_context: ToolContext`，ADK 就會自動注入。
**這個參數不會出現在給模型看的 schema 裡**——模型不知道它存在。

In [8]:
from google.adk.tools import ToolContext


def add_to_cart(item: str, quantity: int, tool_context: ToolContext) -> dict:
    """把商品加入購物車。

    Args:
        item: 商品名稱。
        quantity: 數量。
    """
    cart = tool_context.state.get("cart", [])
    cart = cart + [{"item": item, "quantity": quantity}]
    tool_context.state["cart"] = cart  # 寫回 state
    return {"ok": True, "cart_size": len(cart)}


def view_cart(tool_context: ToolContext) -> dict:
    """查看目前購物車內容。"""
    return {"cart": tool_context.state.get("cart", [])}


# 確認 tool_context 沒有洩漏到 schema
print("模型看到的參數 :", list(
    FunctionTool(func=add_to_cart)._get_declaration().parameters_json_schema["properties"]
))

模型看到的參數 : ['item', 'quantity']


In [9]:
from google.adk.runners import InMemoryRunner

shop = LlmAgent(
    name="shop_bot",
    model=get_model(),
    instruction="你是購物助理。使用者要加購物車就呼叫 add_to_cart，要看就呼叫 view_cart。用繁體中文回答。",
    tools=[add_to_cart, view_cart],
)

shop_runner = InMemoryRunner(agent=shop, app_name="concept_track")
sid = await new_session(shop_runner)

print(await ask(shop_runner, "幫我加兩顆高麗菜", session_id=sid))
print(await ask(shop_runner, "再加一瓶醬油", session_id=sid))
print(await ask(shop_runner, "我車上有什麼？", session_id=sid))

print("\n--- session state ---")
print_state(await peek_state(shop_runner, sid))

好的，已經幫您把 2 顆高麗菜加入購物車了！


好的，已經幫您把 1 瓶醬油加入購物車了！


您的購物車目前有：
- 高麗菜：2 顆
- 醬油：1 瓶

--- session state ---
  cart: [{'item': '高麗菜', 'quantity': 2}, {'item': '醬油', 'quantity': 1}]


`cart` 留在 session state 裡了。這就是 agent 之間傳資料的主要管道，
第 04 章會完整講 state 的作用域與生命週期。

## 6. 內建工具與它的限制

ADK 附了幾個由 Google 端執行的內建工具，最常用的是 `google_search`。
它跟你自己寫的函式工具不同——實際搜尋是在**模型端**做的。

In [10]:
from google.adk.tools import google_search

searcher = LlmAgent(
    name="searcher",
    model=get_model(),
    instruction="你是研究助理。需要最新資訊時使用搜尋，並用繁體中文摘要重點。",
    tools=[google_search],
)

try:
    print(await run_once(searcher, "Google ADK 2.0 有哪些主要變化？請摘要三點。"))
except Exception as exc:
    print(f"⚠️ 搜尋沒跑成：{type(exc).__name__}")
    print(str(exc)[:200])

⚠️ 搜尋沒跑成：_ResourceExhaustedError

On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/google-gemini/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'me


> **⚠️ Search grounding 有自己獨立的配額，而且比一般模型呼叫緊很多。**
>
> 免費層的 Google Search grounding 是**跟模型配額分開計算**的，而且額度小得多。
> 也就是說：`gemini-flash-lite-latest` 還能正常對話，但只要一掛上
> `google_search` 就回 429——**換模型也沒用**，因為卡的是 grounding 配額。
>
> 撞到的話：等配額重置，或到 Google Cloud 開啟計費。
> 本章其他小節都不需要搜尋，可以直接往下讀。

### ⚠️ 限制：內建工具不能跟其他工具混用

這是最容易踩的坑之一——`google_search` 沒辦法跟你自己的函式工具放在同一個
agent 裡。下面實測會發生什麼事：

In [11]:
try:
    mixed = LlmAgent(
        name="mixed",
        model=get_model(),
        instruction="研究助理。",
        tools=[google_search, get_stock_price],
    )
    print(await run_once(mixed, "台積電現在多少錢？"))
except Exception as exc:
    print(f"{type(exc).__name__}:\n{str(exc)[:400]}")
    if "429" in str(exc):
        print("\n（這次是撞到 grounding 配額，不是混用限制本身。"
              "配額恢復後重跑就會看到真正的錯誤訊息。）")

_ResourceExhaustedError:

On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/google-gemini/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your cu

（這次是撞到 grounding 配額，不是混用限制本身。配額恢復後重跑就會看到真正的錯誤訊息。）


**解法**：把用到內建工具的 agent 獨立出來，再用 `AgentTool` 把它包成一個
工具給主 agent 用。這是第 09 章的主題。

## 本章重點

- **一個 Python 函式就是一個工具**，schema 由 ADK 從型別註記 + docstring 自動生成。
- **docstring 是送給模型的 API 文件**，不是給人看的註解。工具沒被正確呼叫，
  先回頭檢查它。
- 用 `FunctionTool(func=f)._get_declaration()` 可以看到模型實際收到什麼。
- **`tool_context: ToolContext`** 讓工具能讀寫 session state，而且不會出現在 schema 裡。
- **內建工具（`google_search`）不能跟自訂工具混在同一個 agent**，
  要靠 `AgentTool` 拆開。
- **Search grounding 有獨立且緊很多的免費配額**，換模型救不了。

## 動手練習

1. 把 `get_current_time` 的 docstring 整段刪掉，重跑第 2 節。模型還會正確呼叫嗎？
2. 幫 `add_to_cart` 加一個 `remove_from_cart` 工具，讓使用者可以說「把醬油拿掉」。
3. 第 4 節的 `analyst` 問它「今天台北天氣如何？順便看一下鴻海股價」，
   用 `trace=True` 觀察它是**一次**呼叫兩個工具，還是分兩輪。

---
**下一站 → `03_models_and_output.ipynb`**：換模型、控制生成參數、
以及讓 agent 吐出結構化的 JSON。